# FreshSight - Freshness Regression Model

Notebook ini disesuaikan dengan dataset lokal yang memiliki struktur VOC-style:
- `dataset/JPEGImages/` — berisi gambar-gambar (*.jpg)
- `dataset/Annotations/` — berisi anotasi XML (Pascal VOC format)

Label freshness:
- `fresh_*` → 0.0 (segar)
- `normal_*` → 0.5 (sedang)
- `rotten_*` → 1.0 (busuk)

Model menggunakan MobileNetV2 (Transfer Learning) untuk regresi freshness index.

## 1. Install Dependencies (jika perlu)
Uncomment jika menjalankan di environment conda lokal.

In [ ]:
# !conda install -c conda-forge tqdm -y
# !conda install -c conda-forge matplotlib -y
# !conda install -c conda-forge pandas -y
# !conda install -c conda-forge opencv -y

## 2. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import os
import cv2
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
from tqdm import tqdm
from random import shuffle

## 3. Konfigurasi Path Dataset

In [ ]:
IMAGES_DIR = os.path.join('dataset', 'JPEGImages')
ANNOTATIONS_DIR = os.path.join('dataset', 'Annotations')
IMG_SIZE = 100  # Ukuran gambar input untuk MobileNetV2

## 4. Fungsi untuk Parsing Anotasi XML

Fungsi ini membaca file XML dan mengekstrak:
- Nama file gambar
- Label objek (fresh_*, normal_*, rotten_*)
- Bounding box (opsional, untuk crop)

Mapping label ke freshness score:
- `fresh_*` → 0.0 (segar/fresh)
- `normal_*` → 0.5 (sedang/medium fresh)
- `rotten_*` → 1.0 (busuk/not fresh)

In [ ]:
def get_freshness_score(label_name):
    """
    Konversi label nama objek ke freshness score.
    fresh_* -> 0.0 (segar)
    normal_* -> 0.5 (sedang)
    rotten_* -> 1.0 (busuk)
    """
    label_lower = label_name.lower()
    if label_lower.startswith('fresh_'):
        return 0.0
    elif label_lower.startswith('normal_'):
        return 0.5
    elif label_lower.startswith('rotten_'):
        return 1.0
    else:
        return None  # Label tidak dikenal


def parse_annotation(xml_path):
    """
    Parse file anotasi XML Pascal VOC.
    Returns: list of dict dengan keys: 'filename', 'label', 'score', 'bbox'
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()
    
    filename = root.find('filename').text
    
    objects = []
    for obj in root.findall('object'):
        name = obj.find('name').text
        score = get_freshness_score(name)
        if score is not None:
            bbox = obj.find('bndbox')
            xmin = int(bbox.find('xmin').text)
            ymin = int(bbox.find('ymin').text)
            xmax = int(bbox.find('xmax').text)
            ymax = int(bbox.find('ymax').text)
            objects.append({
                'filename': filename,
                'label': name,
                'score': score,
                'bbox': (xmin, ymin, xmax, ymax)
            })
    return objects

## 5. Load Semua Anotasi dan Lihat Distribusi Label

In [ ]:
all_annotations = []
xml_files = [f for f in os.listdir(ANNOTATIONS_DIR) if f.endswith('.xml')]

print(f'Total file anotasi XML: {len(xml_files)}')

for xml_file in tqdm(xml_files, desc='Parsing annotations'):
    xml_path = os.path.join(ANNOTATIONS_DIR, xml_file)
    objects = parse_annotation(xml_path)
    all_annotations.extend(objects)

print(f'\nTotal objek yang ditemukan: {len(all_annotations)}')

# Lihat distribusi label
df_annotations = pd.DataFrame(all_annotations)
print('\nDistribusi label:')
print(df_annotations['label'].value_counts())
print('\nDistribusi freshness score:')
print(df_annotations['score'].value_counts())

## 6. Load dan Preprocess Gambar

Untuk setiap objek pada anotasi, crop gambar sesuai bounding box, resize ke 100x100, dan konversi BGR ke RGB.

In [ ]:
def load_data(annotations, images_dir, img_size=100):
    """
    Load gambar-gambar berdasarkan anotasi.
    Untuk setiap objek, crop bounding box dari gambar, resize ke img_size x img_size.
    
    Returns: X (array gambar), Y (array freshness score)
    """
    data = []  # List of (image, score)
    
    for ann in tqdm(annotations, desc='Loading images'):
        img_path = os.path.join(images_dir, ann['filename'])
        if not os.path.exists(img_path):
            continue
        
        img = cv2.imread(img_path)
        if img is None:
            continue
        
        # Crop sesuai bounding box
        xmin, ymin, xmax, ymax = ann['bbox']
        # Pastikan koordinat valid
        h, w = img.shape[:2]
        xmin = max(0, xmin)
        ymin = max(0, ymin)
        xmax = min(w, xmax)
        ymax = min(h, ymax)
        
        cropped = img[ymin:ymax, xmin:xmax]
        if cropped.size == 0:
            continue
        
        cropped = cv2.resize(cropped, (img_size, img_size))
        cropped = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)
        
        data.append([cropped, ann['score']])
    
    # Shuffle data
    print('Shuffling data...')
    shuffle(data)
    
    X = np.array([d[0] for d in data])
    Y = np.array([d[1] for d in data])
    
    return X, Y

In [ ]:
X, Y = load_data(all_annotations, IMAGES_DIR, IMG_SIZE)
print(f'\nShape X: {X.shape}')
print(f'Shape Y: {Y.shape}')
print(f'\nDistribusi Y (freshness score):')
print(pd.Series(Y).value_counts())

## 7. Split Data: Train dan Validation

Bagi data menjadi 80% training dan 20% validation.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Validation set: {X_val.shape[0]} samples')
print(f'\nDistribusi Y_train:')
print(pd.Series(Y_train).value_counts())
print(f'\nDistribusi Y_val:')
print(pd.Series(Y_val).value_counts())

## 8. Visualisasi Sampel Data

In [ ]:
def show_samples(X, Y, n=12):
    """
    Tampilkan beberapa sampel gambar beserta freshness score-nya.
    """
    fig, axes = plt.subplots(2, 6, figsize=(18, 6))
    
    # Pilih indeks acak
    indices = np.random.choice(len(X), min(n, len(X)), replace=False)
    
    score_labels = {0.0: 'FRESH', 0.5: 'NORMAL', 1.0: 'ROTTEN'}
    
    for i, idx in enumerate(indices):
        row = i // 6
        col = i % 6
        axes[row][col].imshow(X[idx])
        label = score_labels.get(Y[idx], f'Score: {Y[idx]:.2f}')
        axes[row][col].set_title(label, fontsize=10)
        axes[row][col].axis('off')
    
    plt.tight_layout()
    plt.show()

show_samples(X_train, Y_train)

## 9. Build Model (MobileNetV2 + Custom Layers)

Arsitektur model:
1. MobileNetV2 (pretrained ImageNet, frozen) sebagai feature extractor
2. BatchNormalization
3. 2x SeparableConv2D (64 filters, relu)
4. BatchNormalization + MaxPooling2D + Dropout(0.4)
5. 2x Conv2D (128 filters, relu)
6. BatchNormalization + MaxPooling2D + Dropout(0.5)
7. Flatten + Dense(128, relu) + Dropout(0.3)
8. Dense(1, sigmoid) — output freshness index (0-1)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, BatchNormalization, SeparableConv2D, MaxPooling2D, Dropout, Flatten, Dense
import keras

In [ ]:
# Load MobileNetV2 pretrained
mobilenetv2_model = tf.keras.applications.MobileNetV2(
    weights='imagenet', 
    include_top=False, 
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Freeze semua layer MobileNetV2
for layer in mobilenetv2_model.layers:
    layer.trainable = False

# Build Sequential model
model = Sequential()

# Base model
model.add(mobilenetv2_model)

# Custom layers
model.add(BatchNormalization())
model.add(SeparableConv2D(64, (3, 3), kernel_initializer='he_uniform', padding='same', activation='relu'))
model.add(SeparableConv2D(64, (3, 3), kernel_initializer='he_uniform', padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D((2, 2)))
model.add(Dropout(0.4))

model.add(Conv2D(128, (3, 3), kernel_initializer='he_uniform', padding='same', activation='relu'))
model.add(Conv2D(128, (3, 3), kernel_initializer='he_uniform', padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D((2, 2)))
model.add(Dropout(0.5))

model.add(Flatten())
model.add(Dense(128, activation='relu', kernel_initializer='he_uniform'))
model.add(Dropout(0.3))

# Output: sigmoid -> 0 (fresh) to 1 (rotten)
model.add(Dense(1, activation='sigmoid'))

# Print summary
model.summary()

## 10. Compile & Train Model

In [ ]:
# Callbacks
lr_rate = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=6, verbose=1, mode='min',
    min_lr=0.00002, cooldown=2
)

check_point = tf.keras.callbacks.ModelCheckpoint(
    filepath='freshness_model_checkpoint.h5', 
    monitor='val_loss', 
    verbose=1, 
    save_best_only=True,
    save_weights_only=False, 
    mode='min'
)

# Compile model
# Karena ini regresi (predict freshness score 0-1), gunakan MSE loss
# Namun notebook asli menggunakan binary_crossentropy (treat sebagai classification-like regression)
# Kita ikuti pendekatan original: binary crossentropy + accuracy
model.compile(
    loss=keras.losses.binary_crossentropy, 
    optimizer=keras.optimizers.Adam(learning_rate=0.001), 
    metrics=['accuracy']
)

In [ ]:
# Normalisasi pixel values ke [0, 1]
X_train_norm = X_train / 255.0
X_val_norm = X_val / 255.0

In [ ]:
# Training
history = model.fit(
    X_train_norm, Y_train,
    batch_size=2,
    validation_data=(X_val_norm, Y_val),
    epochs=10,
    callbacks=[check_point, lr_rate]
)

## 11. Visualisasi Training History

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(history.history['loss'], label='Training Loss')
ax1.plot(history.history['val_loss'], label='Validation Loss')
ax1.set_title('Loss over Epochs')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()

# Accuracy
ax2.plot(history.history['accuracy'], label='Training Accuracy')
ax2.plot(history.history['val_accuracy'], label='Validation Accuracy')
ax2.set_title('Accuracy over Epochs')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()

plt.tight_layout()
plt.show()

## 12. Save Final Model

In [ ]:
model.save('freshness_model_final.h5')
print('Model berhasil disimpan ke freshness_model_final.h5')

## 13. Evaluasi Model pada Validation Set

In [ ]:
# Load best model
best_model = tf.keras.models.load_model('freshness_model_checkpoint.h5')

# Evaluasi
val_loss, val_acc = best_model.evaluate(X_val_norm, Y_val)
print(f'\nValidation Loss: {val_loss:.4f}')
print(f'Validation Accuracy: {val_acc:.4f}')

## 14. Prediksi dan Visualisasi Hasil

In [ ]:
def classify_freshness(score):
    """
    Klasifikasi freshness berdasarkan score prediksi.
    """
    if score < 0.33:
        return 'FRESH'
    elif score < 0.66:
        return 'MEDIUM FRESH'
    else:
        return 'NOT FRESH'


# Prediksi beberapa sampel dari validation set
n_samples = 12
sample_indices = np.random.choice(len(X_val), min(n_samples, len(X_val)), replace=False)

fig, axes = plt.subplots(2, 6, figsize=(20, 7))

for i, idx in enumerate(sample_indices):
    row = i // 6
    col = i % 6
    
    img = X_val[idx]
    true_score = Y_val[idx]
    
    # Prediksi
    pred_score = best_model.predict(np.expand_dims(img / 255.0, axis=0), verbose=0)[0][0]
    
    # Klasifikasi
    pred_class = classify_freshness(pred_score)
    true_class = classify_freshness(true_score)
    
    axes[row][col].imshow(img)
    color = 'green' if pred_class == true_class else 'red'
    axes[row][col].set_title(
        f'True: {true_class}\nPred: {pred_class}\n({pred_score:.3f})',
        fontsize=8, color=color
    )
    axes[row][col].axis('off')

plt.suptitle('Prediksi Freshness pada Validation Set', fontsize=14)
plt.tight_layout()
plt.show()

## 15. Evaluasi Single Image (Opsional)

Gunakan cell ini untuk menguji model pada satu gambar tertentu.

In [ ]:
def evaluate_single_image(model, image_path, img_size=100):
    """
    Evaluasi freshness untuk satu gambar.
    """
    img = cv2.imread(image_path)
    if img is None:
        print(f'Error: Tidak bisa membaca gambar {image_path}')
        return
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (img_size, img_size))
    img_normalized = img_resized / 255.0
    
    # Prediksi
    prediction = model.predict(np.expand_dims(img_normalized, axis=0), verbose=0)[0][0]
    freshness_class = classify_freshness(prediction)
    
    # Tampilkan
    plt.figure(figsize=(6, 6))
    plt.imshow(img_rgb)
    plt.title(f'Prediction: {prediction:.4f}\nClassification: {freshness_class}', fontsize=14)
    plt.axis('off')
    plt.show()
    
    print(f'Freshness Score: {prediction:.4f}')
    print(f'Classification: {freshness_class}')

# Contoh penggunaan (ganti path sesuai gambar yang ingin dievaluasi):
# evaluate_single_image(best_model, 'dataset/JPEGImages/1.jpg')